# 🧠 Exp E: Low-Rank Adaptation (`PyTorch` LoRA $W_0 + \frac{\alpha}{r} B A$) Fine-Tuning on GPU for Analytical SQL Intent & Risk Triage
**Author:** Akmal Khan, PhD  
**Repository:** [github.com/akmalkhaniub/enterprise-banking-ds-platform](https://github.com/akmalkhaniub/enterprise-banking-ds-platform)

### 🎯 Experiment Objective
We implement and benchmark **Low-Rank Adaptation (LoRA)** from first principles in `PyTorch` on GPU, freezing 98%+ of base Transformer weights while training rank-$r$ adapter matrices ($A \in \mathbb{R}^{r \times d_{in}}, B \in \mathbb{R}^{d_{out} \times r}$) across $r \in \{2, 4, 8, 16\}$ on an enterprise **Natural Language Query $\rightarrow$ Analytical SQL Template & Compliance Risk Tier** benchmark.

In [ ]:
import time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active GPU Device: {device}')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# 1. Build Enterprise Domain Corpus (6 Analytical SQL & Compliance Intent Classes, 2,400 Prompts)
INTENT_TEMPLATES = {
    0: ('SQL_WINDOW_VELOCITY_ZSCORE', [
        'compute rolling 24h customer transaction zscore over partition by customer_id',
        'flag midnight high velocity ibft transfers exceeding 3 sigma standard deviations',
        'window function sum amount over partition by account order by timestamp range'
    ]),
    1: ('SQL_RFM_COHORT_CTE', [
        'segment customers by recency frequency monetary volume using duckdb cte',
        'calculate 180 day dormant churn risk accounts and average casa balance',
        'group by customer_id count transactions sum volume date_diff recency'
    ]),
    2: ('SQL_SARIMAX_LIQUIDITY_AGG', [
        'aggregate daily branch deposit inflows outflows and atm cash replenishment demand',
        'forecast 30 day treasury liquidity during salary window and festive spikes',
        'time series stationarity adf test and walk forward mape evaluation'
    ]),
    3: ('SQL_OLS_PEER_BENCHMARK', [
        'benchmark branch efficiency index controlling for staff count atm footprint and opex',
        'compute ols regression residuals actual vs expected net deposit inflow',
        'identify underperforming regional hubs below 94 percent peer efficiency band'
    ]),
    4: ('RISK_AML_STRUCTURING_HOLD', [
        'quarantine sub 2.5 million pkr raast structuring splits near regulatory reporting threshold',
        'trigger biometric step up authentication on anomalous cross border wire drain',
        'isolation forest anomaly score above 78 requires compliance cooling period'
    ]),
    5: ('GOVERNANCE_PII_REDACTION', [
        'mask pakistani cnic national id iban and raw customer account numbers before llm prompt',
        'enforce zero hallucination pydantic schema validation and sha256 audit hash',
        'document statistical model assumptions confidence intervals and known limitations'
    ]),
}

vocab = {}
def tokenize(text, max_len=18):
    tokens = text.lower().split()
    ids = []
    for tok in tokens[:max_len]:
        if tok not in vocab:
            vocab[tok] = len(vocab) + 1
        ids.append(vocab[tok])
    return ids + [0] * (max_len - len(ids))

noise_words = ['please', 'urgent', 'alco', 'report', 'query', 'production', 'table', 'mart', 'dashboard', 'verify', 'audit', 'pipeline']
X_ids, y_labels = [], []
for label, (_, seeds) in INTENT_TEMPLATES.items():
    for i in range(400):
        base = seeds[i % len(seeds)].split()
        # Randomly drop 20% of words & inject domain filler words to make classification non-trivial
        kept = [w for w in base if np.random.rand() > 0.20]
        extra = list(np.random.choice(noise_words, size=3))
        sentence = ' '.join(kept + extra)
        X_ids.append(tokenize(sentence))
        y_labels.append(label)

X_tensor = torch.tensor(X_ids, dtype=torch.long)
y_tensor = torch.tensor(y_labels, dtype=torch.long)
perm = torch.randperm(len(X_tensor))
X_tensor, y_tensor = X_tensor[perm], y_tensor[perm]

split = int(0.8 * len(X_tensor))
train_ds = TensorDataset(X_tensor[:split].to(device), y_tensor[:split].to(device))
X_val, y_val = X_tensor[split:].to(device), y_tensor[split:].to(device)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
print(f'Corpus Size: {len(X_tensor)} prompts | Vocab: {len(vocab)+1} tokens | Classes: {len(INTENT_TEMPLATES)}')

In [ ]:
# 2. Define Base Transformer Encoder & LoRALinear Adapter Layer (W = W0 + (alpha/r) * B @ A)
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r: int = 8, alpha: float = 16.0):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False
        self.r = r
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.randn(r, base_linear.in_features) * (1.0 / math.sqrt(r)))
        self.lora_B = nn.Parameter(torch.zeros(base_linear.out_features, r))

    def forward(self, x):
        base_out = self.base(x)
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T * self.scaling
        return base_out + lora_out

class DomainTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_classes=6):
        super().__init__()
        self.emb = nn.Embedding(vocab_size + 2, d_model, padding_idx=0)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.ffn1 = nn.Linear(d_model, d_model * 4)
        self.ffn2 = nn.Linear(d_model * 4, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, ids):
        x = self.emb(ids)
        q, k, v = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        attn = torch.softmax((q @ k.transpose(-2, -1)) / math.sqrt(x.size(-1)), dim=-1)
        x = self.norm1(x + attn @ v)
        x = self.norm2(x + self.ffn2(torch.relu(self.ffn1(x))))
        pooled = x.mean(dim=1)
        return self.classifier(pooled)

def inject_lora(model: DomainTransformerClassifier, r: int):
    for p in model.parameters():
        p.requires_grad = False
    model.q_proj = LoRALinear(model.q_proj, r=r, alpha=2*r)
    model.v_proj = LoRALinear(model.v_proj, r=r, alpha=2*r)
    model.ffn1 = LoRALinear(model.ffn1, r=r, alpha=2*r)
    model.ffn2 = LoRALinear(model.ffn2, r=r, alpha=2*r)
    for p in model.classifier.parameters():
        p.requires_grad = True
    return model

In [ ]:
# 3. Pre-train Base Backbone Slightly, Then Benchmark Zero-Shot vs LoRA Ranks (r=2, 4, 8, 16) vs Full Fine-Tuning
base_init = DomainTransformerClassifier(len(vocab) + 5).to(device)
# Save initial checkpoint state
init_state = {k: v.clone() for k, v in base_init.state_dict().items()}

def run_finetune_experiment(mode_name, rank=None, epochs=18):
    model = DomainTransformerClassifier(len(vocab) + 5).to(device)
    model.load_state_dict(init_state)
    total_params = sum(p.numel() for p in model.parameters())

    if mode_name == 'Zero-Shot Untuned Backbone':
        trainable_params = 0
        elapsed = 0.0
    else:
        if rank is not None:
            # Warmup embedding layer for 3 epochs then freeze backbone and train only LoRA adapters
            warm_opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
            for _ in range(2):
                for xb, yb in train_loader:
                    warm_opt.zero_grad()
                    nn.CrossEntropyLoss()(model(xb), yb).backward()
                    warm_opt.step()
            model = inject_lora(model, r=rank).to(device)
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=4e-3, weight_decay=1e-4)
        loss_fn = nn.CrossEntropyLoss()
        t0 = time.perf_counter()
        model.train()
        for _ in range(epochs):
            for xb, yb in train_loader:
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                opt.step()
        elapsed = time.perf_counter() - t0

    model.eval()
    with torch.no_grad():
        preds = model(X_val).argmax(dim=-1).cpu().numpy()
    y_true = y_val.cpu().numpy()
    return {
        'Adaptation Strategy': mode_name,
        'Trainable Params': trainable_params,
        'Trainable Share (%)': round(trainable_params / total_params * 100.0, 2),
        'SQL Intent Accuracy (%)': round(accuracy_score(y_true, preds) * 100.0, 2),
        'Macro F1 Score': round(f1_score(y_true, preds, average='macro'), 4),
        'GPU Train Time (s)': round(elapsed, 3)
    }

lora_results = pd.DataFrame([
    run_finetune_experiment('Zero-Shot Untuned Backbone', rank=None, epochs=0),
    run_finetune_experiment('LoRA Rank r=2 Adapter', rank=2),
    run_finetune_experiment('LoRA Rank r=4 Adapter', rank=4),
    run_finetune_experiment('LoRA Rank r=8 Adapter', rank=8),
    run_finetune_experiment('LoRA Rank r=16 Adapter', rank=16),
    run_finetune_experiment('Full Parameter Fine-Tuning (100% Weights)', rank=None),
])
print('=== LORA PARAMETER-EFFICIENT FINE-TUNING (PEFT) BENCHMARK ===')
print(lora_results.to_string(index=False))